# WEEK 7 – DIMENSIONALITY REDUCTION

## Aim
To reduce the dimensionality of the dataset while retaining important information using feature selection and Principal Component Analysis.

## Activities Performed
1. Feature Selection.
2. Variance Thresholding.
3. Correlation-Based Feature Selection.
4. Multicollinearity Analysis using VIF.
5. Principal Component Analysis (PCA).
6. Explained Variance Analysis.
7. PCA Visualization.

## Key Findings
- Seven personality features were selected.
- No features were removed through Variance Thresholding.
- No highly correlated feature pairs were found using the selected threshold of 0.80.
- VIF values were below 5.
- Six principal components retained approximately 95.07% of the variance.

## Outcome
The original seven features were reduced to six principal components while retaining approximately 95.07% of the variance.

## Setup
Run this notebook from top to bottom. The cleaned dataset must be available at `../Dataset/youth_drug_consumption_cleaned.csv`.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from statsmodels.stats.outliers_influence import variance_inflation_factor

cleaned_path = Path('../Dataset/youth_drug_consumption_cleaned.csv')
if not cleaned_path.exists():
    raise FileNotFoundError(
        'Cleaned dataset not found. Run the Week 3 notebook first and save '
        'youth_drug_consumption_cleaned.csv inside the Dataset folder.'
    )

df_clean = pd.read_csv(cleaned_path)
print('Cleaned dataset loaded:', df_clean.shape)


In [ ]:
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from statsmodels.stats.outliers_influence import variance_inflation_factor

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Create a separate dataset
df_dr = df_clean.copy()

# Select personality features
features = [
    "Nscore", "Escore", "Oscore",
    "Ascore", "Cscore", "Impulsive", "SS"
]

X = df_dr[features].copy()

print("Dataset Shape:", X.shape)
print("\nSelected Features:")
print(features)

display(X.head())

## 7.1 Feature Selection

Feature Selection is the process of selecting the most relevant features from a dataset.

It helps remove irrelevant or less informative features, reduce computation time and simplify analysis.

In [ ]:
print("Selected Features:")
for feature in features:
    print(feature)

print("\nTotal Number of Features:", len(features))

## 7.2 Variance Thresholding

Variance Thresholding is a feature selection technique that removes features with very low variance.

Features with almost constant values provide less information for analysis.

A threshold of 0.01 is used to identify low-variance features.

In [ ]:
# Calculate variance
feature_variance = X.var()

print("Variance of Each Feature:")
print(feature_variance)

# Apply Variance Thresholding
selector = VarianceThreshold(threshold=0.01)

X_variance = selector.fit_transform(X)

selected_features = X.columns[
    selector.get_support()
].tolist()

removed_features = X.columns[
    ~selector.get_support()
].tolist()

print("\nRetained Features:")
print(selected_features)

print("\nRemoved Features:")
print(removed_features)

print("\nFeatures Before:", X.shape[1])
print("Features After:", len(selected_features))

In [ ]:
plt.figure(figsize=(10, 5))

sns.barplot(
    x=feature_variance.index,
    y=feature_variance.values
)

plt.axhline(
    y=0.01,
    color="red",
    linestyle="--",
    label="Threshold = 0.01"
)

plt.title("Feature Variance Analysis")
plt.xlabel("Features")
plt.ylabel("Variance")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

## 7.3 Correlation-Based Feature Selection

Correlation-Based Feature Selection identifies relationships between numerical features.

Highly correlated features may contain overlapping information.

A correlation threshold of 0.80 is used to identify strongly correlated feature pairs.

In [ ]:
# Calculate correlation matrix
correlation_matrix = X.corr()

print("Correlation Matrix:")
display(correlation_matrix)

# Visualize correlation
plt.figure(figsize=(10, 7))

sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Personality Features Correlation Heatmap")
plt.tight_layout()
plt.show()

In [ ]:
# Identify highly correlated feature pairs
threshold = 0.80

high_corr_pairs = []

for i in range(len(features)):
    for j in range(i + 1, len(features)):

        corr_value = correlation_matrix.iloc[i, j]

        if abs(corr_value) > threshold:
            high_corr_pairs.append(
                (features[i], features[j], corr_value)
            )

print("Highly Correlated Feature Pairs:")

if high_corr_pairs:
    for pair in high_corr_pairs:
        print(pair)
else:
    print("No highly correlated feature pairs found.")

## 7.4 Multicollinearity Analysis

Multicollinearity occurs when two or more independent features are highly correlated.

Variance Inflation Factor (VIF) is used to identify multicollinearity.

A VIF value greater than 5 is considered a warning sign that a feature needs further investigation.

In [ ]:
# Standardize features before VIF calculation
scaler_vif = StandardScaler()

X_vif = pd.DataFrame(
    scaler_vif.fit_transform(X),
    columns=features
)

# Calculate VIF
vif_data = pd.DataFrame()

vif_data["Feature"] = features

vif_data["VIF"] = [
    variance_inflation_factor(
        X_vif.values, i
    )
    for i in range(X_vif.shape[1])
]

print("Variance Inflation Factor:")
display(vif_data)

print("\nFeatures with VIF greater than 5:")

print(
    vif_data[vif_data["VIF"] > 5]
)

## 7.5 Principal Component Analysis (PCA)

Principal Component Analysis is a dimensionality reduction technique that transforms the original correlated features into a smaller set of uncorrelated variables called principal components.

PCA helps reduce the number of dimensions while retaining as much information as possible.

In [ ]:
# Standardize the original personality features
scaler_pca = StandardScaler()

X_scaled = scaler_pca.fit_transform(X)

# Apply PCA
pca = PCA()

X_pca = pca.fit_transform(X_scaled)

print("Original Number of Features:", X.shape[1])
print("Number of Principal Components:", X_pca.shape[1])

print("\nPCA Completed Successfully!")

## 7.6 Explained Variance Ratio

Explained Variance Ratio indicates how much information or variance each principal component retains from the original dataset.

Cumulative Explained Variance represents the total variance retained by the selected components.

In [ ]:
explained_variance = pca.explained_variance_ratio_

cumulative_variance = np.cumsum(
    explained_variance
)

variance_table = pd.DataFrame({
    "Principal Component": [
        f"PC{i+1}" for i in range(len(explained_variance))
    ],
    "Explained Variance": explained_variance,
    "Cumulative Variance": cumulative_variance
})

display(variance_table)

# Number of components needed for 95% variance
n_components = np.argmax(
    cumulative_variance >= 0.95
) + 1

print(
    "\nComponents needed to retain 95% variance:",
    n_components
)

## 7.7 Explained Variance Visualization

A scree plot is used to visualize the explained variance of each principal component.

A cumulative variance plot helps identify the number of components needed to retain a desired percentage of information.

In [ ]:
components = range(1, len(explained_variance) + 1)

plt.figure(figsize=(10, 5))

plt.plot(
    components,
    explained_variance,
    marker="o",
    label="Individual Variance"
)

plt.plot(
    components,
    cumulative_variance,
    marker="s",
    label="Cumulative Variance"
)

plt.axhline(
    y=0.95,
    linestyle="--",
    label="95% Variance"
)

plt.xlabel("Principal Components")
plt.ylabel("Explained Variance Ratio")
plt.title("PCA Explained Variance")
plt.xticks(list(components))
plt.legend()
plt.grid()
plt.tight_layout()
plt.show()

## 7.8 Dimensionality Reduction

The original seven personality features are reduced to the number of principal components required to retain at least 95% of the variance.

In [ ]:
# Apply PCA with selected components
pca_reduced = PCA(
    n_components=n_components
)

X_reduced = pca_reduced.fit_transform(
    X_scaled
)

# Create reduced dataset
pca_columns = [
    f"PC{i+1}" for i in range(n_components)
]

df_pca = pd.DataFrame(
    X_reduced,
    columns=pca_columns
)

print("Original Dataset Shape:", X.shape)
print("Reduced Dataset Shape:", df_pca.shape)

display(df_pca.head())

## 7.9 Two-Dimensional PCA Visualization

The first two principal components are plotted in a scatter plot to visualize the distribution of observations in reduced dimensions.

Gender is used to distinguish observations in the visualization.

In [ ]:
# Use the first two components from full PCA
pca_2d = X_pca[:, :2]

pca_plot = pd.DataFrame(
    pca_2d,
    columns=["PC1", "PC2"]
)

# Add gender labels
pca_plot["Gender"] = df_clean["Gender"].map({
    -0.48246: "Male",
     0.48246: "Female"
}).values

plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=pca_plot,
    x="PC1",
    y="PC2",
    hue="Gender",
    alpha=0.7
)

plt.title("PCA - Two Dimensional Visualization")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend(title="Gender")
plt.tight_layout()
plt.show()

## 7.10 Experiment Conclusion

Dimensionality reduction was performed on the personality features of the drug consumption dataset.

Variance Thresholding was used to identify low-variance features. Correlation analysis and VIF were used to examine relationships and multicollinearity. Principal Component Analysis was applied to transform the original features into principal components.

Explained variance and PCA scatter plots were used to visualize the results. This process helped understand how dimensionality reduction can simplify data while retaining important information.

In [ ]:
print("EXPERIMENT 6 - FINAL SUMMARY")

print("\nOriginal Dataset Shape:", X.shape)

print("Features Before Selection:", len(features))

print("Features After Variance Thresholding:",
      len(selected_features))

print("Total PCA Components Generated:", X_pca.shape[1])

print("Components Selected for 95% Variance:",
      n_components)

print("Variance Retained:",
      round(cumulative_variance[n_components-1] * 100, 2), "%")

print("Reduced Dataset Shape:", df_pca.shape)

print("\nExperiment 6 Completed Successfully!")